# Data preprocessing


In [20]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt


In [21]:
dataset = fetch_ucirepo(id=336)

df_raw = dataset.data.original

df_raw



,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,...,47.0,6700.0,4.9,no,no,no,good,no,no,notckd
396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,...,54.0,7800.0,6.2,no,no,no,good,no,no,notckd
397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,...,49.0,6600.0,5.4,no,no,no,good,no,no,notckd
398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,...,51.0,7200.0,5.9,no,no,no,good,no,no,notckd


In [22]:
# Dataset name
dataset.metadata.name

'Chronic Kidney Disease'

In [23]:
# 1. Define required target columns mapping (UCI raw names -> target assignment names)
column_mapping = {
    "age": "age",
    "bp": "blood pressure",
    "sg": "specific gravity",
    "al": "albumin",
    "su": "sugar",
    "bgr": "blood glucose random",
    "bu": "blood urea",
    "sod": "sodium",
    "pot": "potassium",
    "hemo": "hemoglobin",
    "pcv": "packed cell volume",
    "wbcc": "white blood cell count",
    "rbcc": "red blood cell count",
    "class": "class",
}

In [24]:
# Keep only requested columns and rename
df = df_raw[list(column_mapping.keys())].rename(columns=column_mapping)

In [25]:
df

,age,blood pressure,specific gravity,albumin,sugar,blood glucose random,blood urea,sodium,potassium,hemoglobin,packed cell volume,white blood cell count,red blood cell count,class
0,48.0,80.0,1.020,1.0,0.0,121.0,36.0,NaN,NaN,15.4,44.0,7800.0,5.2,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,18.0,NaN,NaN,11.3,38.0,6000.0,NaN,ckd
2,62.0,80.0,1.010,2.0,3.0,423.0,53.0,NaN,NaN,9.6,31.0,7500.0,NaN,ckd
3,48.0,70.0,1.005,4.0,0.0,117.0,56.0,111.0,2.5,11.2,32.0,6700.0,3.9,ckd
4,51.0,80.0,1.010,2.0,0.0,106.0,26.0,NaN,NaN,11.6,35.0,7300.0,4.6,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,140.0,49.0,150.0,4.9,15.7,47.0,6700.0,4.9,notckd
396,42.0,70.0,1.025,0.0,0.0,75.0,31.0,141.0,3.5,16.5,54.0,7800.0,6.2,notckd
397,12.0,80.0,1.020,0.0,0.0,100.0,26.0,137.0,4.4,15.8,49.0,6600.0,5.4,notckd
398,17.0,60.0,1.025,0.0,0.0,114.0,50.0,135.0,4.9,14.2,51.0,7200.0,5.9,notckd


In [26]:
# 2. Clean numeric strings containing '?' or trailing tabs/spaces (common in UCI CKD dataset)
numeric_cols = [
    "age",
    "blood pressure",
    "specific gravity",
    "albumin",
    "sugar",
    "blood glucose random",
    "blood urea",
    "sodium",
    "potassium",
    "hemoglobin",
    "packed cell volume",
    "white blood cell count",
    "red blood cell count",
]

In [27]:
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.strip().replace("?", np.nan), errors="coerce"
    )

# 3. Convert Hemoglobin from g/dl to g/l (* 10)
df["hemoglobin"] = df["hemoglobin"] * 10

In [28]:
# 4. Cleans and Recodes 'class' column:
# Raw values: 'ckd' (affected) / 'notckd' (control) -> Recoded as 'a' / 'c'
df["class"] = (
    df["class"]
    .astype(str)
    .str.strip()
    .map({"ckd": "a", "ckd\t": "a", "notckd": "c"})
)

In [29]:
df_modified = df.dropna(thresh=df.shape[1] - 2).copy()

In [30]:
print(f"Original row count: {len(df_raw)}")
print(f"Modified DataFrame row count: {len(df_modified)}")
print(f"Rows dropped: {len(df_raw) - len(df_modified)}")

Original row count: 400
Modified DataFrame row count: 265
Rows dropped: 135
